In [6]:
import random
random.seed(0)
import numpy as np
np.random.seed(0)
import tensorflow as tf
tf.random.set_seed(0)

In [7]:
import os
import json
from zipfile import ZipFile
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers,models

In [8]:
!pip install kaggle

In [9]:
kaggle_credentials=json.load(open('/content/kaggle (1).json'))

In [10]:
os.environ['KAGGLE_USERNAME']=kaggle_credentials['username']
os.environ['KAGGLE_KEY']=kaggle_credentials['key']


In [11]:
!kaggle datasets download abdallahalidev/plantvillage-dataset

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0


In [12]:
!ls

'kaggle (1).json'   plantvillage-dataset.zip   sample_data


In [13]:
with ZipFile ('plantvillage-dataset.zip','r') as zip_ref:
  zip_ref.extractall()

In [ ]:
print(os.listdir("plantvillage dataset"))
print(len(os.listdir("plantvillage dataset/color")))
print(os.listdir("plantvillage dataset/color")[:5])

['color', 'grayscale', 'segmented']
38
['Corn_(maize)___Northern_Leaf_Blight', 'Tomato___Tomato_mosaic_virus', 'Orange___Haunglongbing_(Citrus_greening)', 'Corn_(maize)___healthy', 'Blueberry___healthy']


Since we are trying to do binary classification as healthy or diseased
We will  have to split the directory into two then ony we can do image processing

In [ ]:
base_dir='plant_disease_binary'
healthy_dir=os.path.join(base_dir,'healthy')
diseased_dir=os.path.join(base_dir,'diseased')
os.makedirs(healthy_dir,exist_ok=True)
os.makedirs(diseased_dir,exist_ok=True)

In [16]:
root_dir="/content/plantvillage dataset/color"

In [ ]:
import shutil
for folder in os.listdir(root_dir): 
  path=os.path.join(root_dir,folder)
  if 'healthy' in folder:  
    for file in os.listdir(path):
      shutil.copy(os.path.join(path,file),healthy_dir) 
  else:
    for file in os.listdir(path):
      shutil.copy(os.path.join(path,file),diseased_dir)

In [ ]:

healthy_pics=0
diseased_pics=0
for image in os.listdir(healthy_dir):
  healthy_pics+=1
for image in os.listdir(diseased_dir):
  diseased_pics+=1
print(healthy_pics)
print(diseased_pics)


15084
39221


Since number of healthy and diseased images are different we will standardise them by undersampling

In [ ]:
import random
diseased_files = os.listdir(diseased_dir)
random.shuffle(diseased_files)
diseased_files = diseased_files[:15084]  

In [ ]:
train_data = train_datagen.flow_from_directory(
    base_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)
train_data = train_datagen.flow_from_directory(
    base_dir,           
    target_size=(128, 128),  
    batch_size=32,          
    class_mode='binary',    
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    base_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

print(train_data.class_indices)


Found 43445 images belonging to 2 classes.
Found 10860 images belonging to 2 classes.
{'diseased': 0, 'healthy': 1}


In [25]:
from tensorflow import keras

In [ ]:
model=keras.Sequential([keras.layers.Flatten(input_shape=(128,128,3)),keras.layers.Dense(128,activation='relu'),keras.layers.Dense(1,activation='sigmoid')])

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)